# FluidFlower Task 1: Full-feature temporal surrogate

This notebook loads in the data for two laboratory CO2 storage experiments and metadata. The experiments have been performed in [FluidFlower rigs](https://fluidflower.w.uib.no/) containing several sand layers.

The model learns how to move a full spatial state forward in time, first with one-step prediction and then with autoregressive rollout.

## Benchmark goal

Given an observed state at time $t$, predict the future state at a chosen horizon $T$.
In practice, we start from one snapshot, predict the next snapshot, and repeat the same learned map until we reach the target time.

## Dataset snapshot

The data contains dense interpretations of saturation and total mass of CO2, provided as snapshots in time, using a fixed Cartesian grid of f 368 x 220 cells (uniform cell-size of 0.0025 cm).
- Each snapshot is stored as a CSV file with time indicated in the file name in hours_minute format.
- Between 0 and 2.5 hours snapshots are provided every 2nd minute.
- Between 2.5 and 12 hours snapshots are provided every 10th minute.
- Between 12 and 72 hours snapshots are provided every 30th minute.
- Every row contains one grid location and the associated physical fields.
- The grid coordinates stay fixed through time, so the task is temporal prediction on a fixed spatial mesh.

## Features (CSV dataformat)

Each row contains two coordinate columns and two state variables:

- `x [m]`, `y [m]`: fixed spatial coordinates
- `saturation [-]`
- `tmCO2 [kg]`

The main and target feature is `tmCO2` describing the total mass of CO2 in the cell with cell center (`x`,`y`) given in Cartesian Coordinates.

## Experiments AC14 and BC02

The two datasets are reanalyzed photographic data from [Haugen et al.](https://www.bing.com/search?pglt=673&q=haugen+et+al+variability&PC=U531&cvid=7db76131cbe0473ea292246f6f2788e9&FORM=ANNAB1) using the Darcy-scale image analysis toolbox [DarSIA](https://link.springer.com/article/10.1007/s11242-023-02000-9). 

As in the reference article, the experiments are referred to as AC14 and BC02. The two geometries are conceptually similar and are on the same meter scale and their geometries consist of the same sands yet differently layered, the same fluids have been used, and the laboratory conditions are comparable.

## Metadata AC14

The metadata is defines below in the code and briefly described here.

The geometry is structured in facies:

![](../doc/img/image2.png)

The geometry has the following dimensions:
- width: 0.90 m
- height: 0.49 m
- depth: 0.01 m

The laboratory conditions have been:
- atmospheric pressure: 1.013 bar
- temperature: 23 deg C

The injection of CO2 is performed in two locations port 1 at coordinates (0.45, 0.03) and port 2 at coordinates (0.72, 0.18). The injection rate is varying slightly over time using a ramp up and down due to experimental limitations. The injection protocol is part of the dataset and provided as csv file. Here for convenience:

|port|duration_s|rate_kg_per_s|mass_kg|
|----|----|----|----|
|1|60|3.12e-09|1.872e-07|
|1|60|6.24e-09|3.744e-07|
|1|60|1.56e-08|9.36e-07|
|1|60|3.12e-08|1.872e-06|
|1|60|4.68e-08|2.808e-06|
|1|2700|6.24e-08|1.6848e-04|
|1|60|4.68e-08|2.808e-06|
|1|60|3.12e-08|1.872e-06|----|
|1|60|1.56e-08|9.36e-07|
|1|60|6.24e-09|3.744e-07|
|1|60|3.12e-09|1.872e-07|
|2|60|3.12e-09|1.872e-07|
|2|60|6.24e-09|3.744e-07|
|2|60|1.56e-08|9.36e-07|
|2|60|3.12e-08|1.872e-06|
|2|60|4.68e-08|2.808e-06|
|2|4500|6.24e-08|2.808e-04|
|2|60|4.68e-08|2.808e-06|
|2|60|3.12e-08|1.872e-06|
|2|60|1.56e-08|9.36e-07|
|2|60|6.24e-09|3.744e-07|
|2|60|3.12e-09|1.872e-07|


## Metadata BC02

The metadata is defines below in the code and briefly described here.

The geometry is structured in facies:

![](../doc/img/image3.png)

The geometry has the following dimensions:
- width: 0.92 m
- height: 0.55 m
- depth: 0.01 m

The laboratory conditions have been:
- atmospheric pressure: 0.995 bar
- temperature: 23 deg C

The injection of CO2 is performed in two locations port 1 at coordinates (0.74, 0.07) and port 2 at coordinates (0.23, 0.24). The injection rate is varying slightly over time using a ramp up and down due to experimental limitations. The injection protocol is part of the dataset and provided as csv file. Here for convenience:

|port|duration_s|rate_kg_per_s|mass_kg|
|----|----|----|----|
|1|60|3.12e-08|1.872e-06|
|1|60|4.68e-08|2.808e-06|
|1|3793|6.24e-08|2.366832e-04|
|1|60|4.68e-08|2.808e-06|
|1|60|3.12e-08|1.872e-06|
|1|60|1.56e-08|9.36e-07|
|2|60|3.12e-08|1.872e-06|
|2|60|4.68e-08|2.808e-06|
|2|4372|6.24e-08|2.728128e-04|
|2|60|4.68e-08|2.808e-06|
|2|60|3.12e-08|1.872e-06|
|2|60|1.56e-08|9.36e-07|

## Hydraulic properties

The geometries of both experiments are based on the same four sand types, and a water layer on top, with the following parameters:

|id|name|porosity|permeability|swi CO2|krel_gas|1-Sg|krel_water|pc [mbar]|
|----|----|----|----|----|----|----|----|----|
|0|water|Nan|Nan|Nan|Nan|Nan|Nan|Nan|
|1|ESF|0.44|4.40E+01|0.32|0.09|0.86|0.71|15|
|2|C|0.44|473|0.14|0.05|0.9|0.93|3.3|
|4|E|0.45|2005|0.12|0.1|0.94|0.93|0.26|
|5|F|0.44|4295|0.12|0.11|0.87|0.72|0.1|

The spatial distribution of the different facies is provided as part of the dataset.

## Notation

Let $s_t$ denote the full spatial state at time $t$. In this notebook, one step of the surrogate is written as

$$
s_{t+\Delta t} = f_{\theta}(s_t, \Delta t)
$$

where $\Delta t$ is the time gap between two snapshots and $f_{\theta}$ is a learned regression map.
When we apply the same map repeatedly, we get an autoregressive rollout:

$$
s_{t_{k+1}} = f_{\theta}(s_{t_k}, t_{k+1} - t_k)
$$


## What this notebook shows

- how to load the csv files
- how to export predictions into the benchmark CSV format
- a simple DMD model for replaying the datasets


## Loading metadata for AC14
The metadata is store in a json file (dimensions, pressure, temperature) and csv file (injection protocol).

In [ ]:
import json
from pathlib import Path

# Folder to data of AC14
ac14_folder = Path("/home/jovyan/shared/folder/data/ac14")

metadata_ac14 = json.load(open(ac14_folder / "metadata.json", "r"))
width_ac14 = metadata_ac14["width"]
height_ac14 = metadata_ac14["height"]
depth_ac14 = metadata_ac14["depth"]
pressure_ac14 = metadata_ac14["pressure"]
temperature_ac14 = metadata_ac14["temperature"]


## Loading the AC14 data
As an example we load the csv file corresponding to the approx. the end of the injection phase of AC14, after 2:30 hours. We extract the mass and convert the data to a displayabe image.

In [ ]:
import matplotlib.pyplot as plt
from ml4gcs.fluidflower_utils.read_csv import load, save, to_2d_array, to_1d_array, TIMESTAMPS, SELECTED_TIMESTAMPS

# Temporary folder for plotting
plt_folder = Path("plots_fludflower_ac14")
plt_folder.mkdir(exist_ok=True)

# Read facies file
facies = load(ac14_folder / "facies.csv", "facies")
facies_2d = to_2d_array(*facies)

plt.figure(figsize=(6, 3))
plt.imshow(facies_2d, origin='lower', extent=[0, width_ac14, 0, height_ac14], cmap='tab20')
plt.colorbar(label='Facies')
plt.title('FluidFlower AC14 - Facies')
plt.xlabel('Width (m)')
plt.ylabel('Height (m)')
plt.grid(False)
plt.tight_layout()
plt.savefig(plt_folder / "fluidflower_ac14_facies.png")
plt.show()
display(Image(filename=str(plt_folder / "fluidflower_ac14_facies.png")))

# Read snapshots and plot
selected_snapshots = [load(ac14_folder / f"ml4gcs_spatial_map_{timestamp}.csv") for timestamp in SELECTED_TIMESTAMPS]
selected_snapshots_2d = [to_2d_array(*snapshot) for snapshot in selected_snapshots]

for timestamp, snapshot in zip(SELECTED_TIMESTAMPS, selected_snapshots_2d):
    plt.figure(figsize=(6, 3))
    plt.imshow(snapshot, origin='lower', extent=[0, width_ac14, 0, height_ac14], cmap='viridis')
    plt.colorbar(label='Saturation')
    plt.title(f'FluidFlower AC14 - Saturation at {timestamp.replace("_", ":")}')
    plt.xlabel('Width (m)')
    plt.ylabel('Height (m)')
    plt.grid(False)
    plt.tight_layout()
    plt.savefig(plt_folder / f"fluidflower_ac14_saturation_{timestamp}.png")
    plt.show()
    display(Image(filename=str(plt_folder / f"fluidflower_ac14_saturation_{timestamp}.png")))